# Helm

Helm paketiert Kubernetes-Manifeste. Ein Chart enthält Templates und Values; ein Release ist eine installierte Instanz. Jede Änderung erzeugt eine Release-Revision. Repositories stellen Charts bereit.

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Chart untersuchen

Chart.yaml beschreibt das Paket; values.yaml liefert Standardwerte. Templates referenzieren diese über .Values und den Release-Namen über .Release.Name.

In [ ]:
%%bash
set -eo pipefail
helm version
cat hostinfo/Chart.yaml hostinfo/values.yaml
cat hostinfo/templates/deployment.yaml hostinfo/templates/service.yaml

In [ ]:
%%bash
set -eo pipefail
helm lint ./hostinfo
helm template training-hostinfo ./hostinfo

## 2. Release installieren

Helm rendert und installiert die Ressourcen. --wait wartet auf deren Bereitschaft. Port 9000 muss frei sein.

In [ ]:
%%bash
set -eo pipefail
helm install training-hostinfo ./hostinfo --wait --timeout 3m
helm list
kubectl get deployment training-hostinfo
kubectl get pods -l app=training-hostinfo
kubectl get svc training-hostinfo

## 3. Upgrade und Skalierung

--set überschreibt einen Value. Die Replikazahl ändert sich, das Pod-Template bleibt gleich.

In [ ]:
%%bash
set -eo pipefail
helm upgrade training-hostinfo ./hostinfo --set replicaCount=5 --wait --timeout 3m
helm history training-hostinfo
kubectl get deployment training-hostinfo

## 4. Rolling Update mit Helm

Eine Annotation im Pod-Template wird über einen Value geändert. Dadurch erzeugt das Deployment ein neues ReplicaSet, ohne dass ein neues Image erforderlich ist.

In [ ]:
%%bash
set -eo pipefail
helm upgrade training-hostinfo ./hostinfo --set replicaCount=5 --set-string demoRevision=2 --wait --timeout 3m
kubectl rollout status deployment/training-hostinfo --timeout=180s
kubectl get rs -l app=training-hostinfo
helm history training-hostinfo

## 5. Rollback

Helm stellt die Konfiguration der Revision 1 wieder her und erzeugt eine neue Revision.

In [ ]:
%%bash
set -eo pipefail
helm rollback training-hostinfo 1 --wait --timeout 3m
helm history training-hostinfo
kubectl get deployment training-hostinfo

## 6. Externes Chart-Repository (optional)

Chart-Repositories erlauben Suche und Bezug externer Charts. Hier wird nichts im Cluster installiert. Internetzugriff erforderlich.

In [ ]:
%%bash
set -eo pipefail
helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
helm repo update prometheus-community
helm search repo prometheus-community/kube-prometheus-stack

## 7. Aufräumen

Das Release und seine Ressourcen entfernen. Das optionale Repository bleibt lokal registriert.

In [ ]:
%%bash
set -eo pipefail
helm uninstall training-hostinfo
helm list
kubectl get deployment,svc training-hostinfo --ignore-not-found

## Fazit

Helm verwaltet parametrisierte Anwendungen und ihre Release-Historie. Rolling Updates bleiben eine Funktion des Kubernetes-Deployments.